# 02 · AgentCore Runtime — put your agent in production
### local test → configure → launch → invoke → sessions → under the hood

This is the service everything else hangs off. **Runtime** is a serverless host for *your* agent code: it builds the container, provisions the IAM role, the ECR repo, autoscaling, networking, and CloudWatch — and runs each user's session in an isolated **microVM** that can live up to **8 hours**.

The contrast that sells it:

| | Traditional (ECS/EKS) | AgentCore Runtime |
|---|---|---|
| You configure | clusters, ALB, autoscaling, IAM, ECR, CloudWatch, VPC | a Python file |
| Time to production | ~2–3 days | ~a few minutes, one command |
| Session model | you build it | isolated microVM per session, up to 8h |

```mermaid
flowchart LR
    A[Your agent code<br/>app.py] -->|configure + launch| B[CodeBuild builds<br/>ARM64 container]
    B --> C[ECR image]
    C --> D[AgentCore Runtime<br/>+ versioned endpoint]
    D --> E[invoke: isolated<br/>microVM per session]
```

> Prereqs (in addition to notebook 00): attach **`BedrockAgentCoreFullAccess`** + **`AmazonBedrockFullAccess`** to your principal. For the CLI path you'll also want **Node.js 20+**. Region stays `us-east-1`.


## A fork in the road (read this — it's the one thing that confuses people in 2026)

There are **two** tools that deploy to Runtime. Both work; they're for different moments:

- **Starter Toolkit** (`bedrock-agentcore-starter-toolkit`) — a **Python API** (`Runtime().configure/launch/invoke`). It runs **from inside this notebook**, which is why we use it for teaching. AWS now labels it *legacy* and is migrating samples away from it, but it is fully functional today.
- **AgentCore CLI** (`@aws/agentcore`, an npm package) — the **recommended** project-based workflow (`agentcore create/dev/deploy/invoke`). It scaffolds a real project directory, so it lives **outside** a notebook. We cover it near the end as the path you'll actually ship with.

We'll deploy with the toolkit (notebook-friendly), then show the exact CLI equivalents. No pretending there's one canonical path.


## 1. The contract: every Runtime agent is just an HTTP server

Under all the abstraction, an AgentCore Runtime agent is a container exposing two endpoints on **port 8080**:

- `POST /invocations` — receives your JSON payload, returns the agent's response.
- `GET /ping` — health check the platform polls.

`BedrockAgentCoreApp` implements both for you. You only write the **entrypoint**: a function that takes the parsed `payload` dict and returns a dict/string (or **yields** for streaming).

The `payload` keys are **your contract** — there's no required schema. We'll use `{"prompt": "..."}`.


In [ ]:
# Local sanity import (no AWS yet) — confirms the runtime SDK is present
from bedrock_agentcore import BedrockAgentCoreApp
_app = BedrockAgentCoreApp()
print("BedrockAgentCoreApp ready. Serves POST /invocations + GET /ping on :8080 when run.")

## 2. Write `app.py` — wrap the Strands agent from notebook 01

We take the exact `flight_agent` and make it deployable. The only additions over notebook 01 are the three highlighted lines: import the app, decorate an entrypoint, and `app.run()` for local serving.

`%%writefile` writes this cell to disk as `app.py` so the toolkit can package it.


In [ ]:
%%writefile app.py
"""TravelMind flight-status agent, packaged for AgentCore Runtime."""
from bedrock_agentcore import BedrockAgentCoreApp          # <-- 1) the runtime app
from strands import Agent, tool

MODEL_HAIKU = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

FLIGHTS = {
    "BA117": {"flight": "BA117", "status": "Delayed", "from": "LHR", "to": "JFK", "delay_min": 75},
    "AA100": {"flight": "AA100", "status": "On time", "from": "JFK", "to": "LHR", "delay_min": 0},
    "AI191": {"flight": "AI191", "status": "Cancelled", "from": "DEL", "to": "SFO", "delay_min": None},
}

@tool
def get_flight_status(flight_number: str) -> dict:
    """Look up the current status of a flight by its flight number (e.g. BA117)."""
    return FLIGHTS.get(flight_number.upper().replace(" ", ""), {"status": "Unknown flight"})

agent = Agent(
    model=MODEL_HAIKU,
    tools=[get_flight_status],
    system_prompt="You are TravelMind, a flight status assistant. Use the tool, then answer plainly.",
)

app = BedrockAgentCoreApp()                                # <-- 2) create the app

@app.entrypoint                                            # <-- 3) this fn handles /invocations
def invoke(payload):
    user_message = payload.get("prompt", "")
    result = agent(user_message)
    return {"result": str(result)}

if __name__ == "__main__":
    app.run()                                              # serve locally on :8080

**Streaming + context (optional):** if your entrypoint **`yield`**s strings instead of returning, AgentCore streams them as server-sent events. And if you declare a second parameter (`def invoke(payload, context):`), the framework passes a `RequestContext` (session id, headers) — useful for per-user logic. The simple `return` form above is all you need to start.


## 3. Test locally *before* you deploy

Deploying to debug is slow. Run the agent on your machine first — it's the same container contract, just local.

```bash
# Terminal 1 — run the agent (serves http://localhost:8080)
python app.py

# Terminal 2 — health check, then a real invocation
curl http://localhost:8080/ping
curl -X POST http://localhost:8080/invocations \
     -H "Content-Type: application/json" \
     -d '{"prompt": "Is BA117 on time? If delayed, by how much?"}'
```

If `/ping` returns healthy and `/invocations` returns the agent's answer, your code is correct. Everything after this is *infrastructure*, which AgentCore handles.


## 4. Deploy from the notebook — `configure` → `launch`

`configure` records *how* to package the agent; `launch` actually builds and ships it.

**`configure` params that matter:**
- `entrypoint="app.py"` — the file with your `@app.entrypoint`.
- `agent_name=...` — a name for the runtime.
- `requirements=[...]` — what to `pip install` into the container (or `requirements_file="requirements.txt"`).
- `auto_create_execution_role=True` — let the toolkit create the IAM role the agent runs as (otherwise pass your own `execution_role=`).
- `auto_create_ecr=True` — let it create the ECR repo for the image.
- `region="us-east-1"`.

> This **provisions real AWS resources and incurs cost**. Run it when you're ready, not on every notebook pass.


In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

runtime = Runtime()

config = runtime.configure(
    entrypoint="app.py",
    agent_name="travelmind_flight",
    requirements=["strands-agents", "bedrock-agentcore"],
    auto_create_execution_role=True,   # toolkit creates the IAM execution role
    auto_create_ecr=True,              # toolkit creates the ECR repository
    region="us-east-1",
)
print("Configured:", config)

In [ ]:
# Build (ARM64 via CodeBuild) -> push to ECR -> create runtime + endpoint. Takes a few minutes.
launch_result = runtime.launch()
print(launch_result)            # contains the agent runtime ARN you'll invoke

### What `launch()` just did for you (and what you'd otherwise hand-build)

- **ARM64 container** built in **AWS CodeBuild** — no local Docker required.
- **ECR repository** created and the image pushed.
- **IAM execution role** + CodeBuild role provisioned with sane policies.
- **Runtime + a versioned endpoint** created and made ready.
- **Autoscaling, networking, CloudWatch** wired automatically.

That list is exactly the ECS/EKS checklist teams normally spend days on. Here it's a side effect of one call.


## 5. Invoke the deployed agent


In [ ]:
resp = runtime.invoke({"prompt": "Is BA117 on time? If delayed, by how much?"})
print(resp)

## 6. Multi-turn: sessions and the microVM

Pass a **`session_id`** and AgentCore routes those calls to the **same warm microVM**. Because the Python process (and your `agent` object) stays warm for that session, the agent's in-process conversation **persists across calls within the session** — no extra setup.

Cross-session, or after the session idles out, that memory is gone. Durable, queryable memory across sessions is a separate service — **notebook 03**.


In [ ]:
# A session id must be reasonably long (>= 33 chars) when it reaches the raw API.
session = "travelmind-session-0001-" + "a" * 16   # comfortably over 33 chars

runtime.invoke({"prompt": "My flight is BA117."}, session_id=session)
followup = runtime.invoke({"prompt": "So is it on time?"}, session_id=session)  # remembers BA117
print(followup)

## 7. Under the hood — the raw call the toolkit wraps

After deployment, a client app doesn't use the toolkit — it calls the **`bedrock-agentcore`** data-plane API directly. Seeing it removes the "magic":

- service name is **`bedrock-agentcore`** (data plane). Control-plane create/list ops live under **`bedrock-agentcore-control`** — easy to mix up.
- `payload` is **bytes** (a blob), so you `json.dumps(...).encode()`.
- `runtimeSessionId` must be **≥ 33 characters**, or you get a validation error.
- the response body is a **stream** — `.read()` it, then decode/parse.


In [ ]:
import boto3, json

AGENT_RUNTIME_ARN = launch_result.agent_arn if hasattr(launch_result, "agent_arn") else "<paste-arn-from-launch_result>"

acr = boto3.client("bedrock-agentcore", region_name="us-east-1")   # DATA plane
resp = acr.invoke_agent_runtime(
    agentRuntimeArn=AGENT_RUNTIME_ARN,
    runtimeSessionId="travelmind-session-0001-" + "a" * 16,        # >= 33 chars
    contentType="application/json",
    accept="application/json",
    payload=json.dumps({"prompt": "Is AA100 on time?"}).encode("utf-8"),
)
print("status:", resp["statusCode"])
print("body  :", resp["response"].read().decode("utf-8"))          # response is a stream

## 8. The recommended production path — the AgentCore CLI

For real projects, AWS now points to the **`@aws/agentcore`** CLI. It scaffolds a project, runs it locally, deploys, and bolts on capabilities. These are **shell** commands (run in a terminal, not this kernel):

```bash
npm install -g @aws/agentcore          # needs Node.js 20+

agentcore create                       # wizard: framework (Strands/LangGraph/...) + Python/TS
cd my-agent
agentcore dev                          # local dev server, hot-reload, local invoke endpoint
agentcore deploy                       # build + ship to AgentCore Runtime
agentcore invoke                       # test the deployed agent

# add capabilities later, then re-deploy:
agentcore add memory
agentcore add identity
agentcore add evaluator                # LLM-as-a-judge
agentcore deploy
```

**Migration note (so stale tutorials don't trip you up):** the official samples repo is moving from the starter toolkit to this CLI. Toolkit-based samples now live in a `legacy/` folder pending migration. The *concepts* are identical — `configure/launch/invoke` ≈ `create/deploy/invoke`. We taught the toolkit because it runs in-cell; ship with the CLI.


## 9. Common Runtime errors → fixes

| Symptom | Cause | Fix |
|---|---|---|
| Build fails / "exec format error" at runtime | Built an x86 image locally | Let `launch()` / CLI build **ARM64** via CodeBuild; don't hand-build x86 |
| `ValidationException` on invoke | `runtimeSessionId` < 33 chars | Use a session id ≥ 33 chars |
| `UnknownService` / op not found in boto3 | Mixed up the service names | Data plane = `bedrock-agentcore`; control plane = `bedrock-agentcore-control` |
| Re-deploy ignores your changes | Stale `.bedrock_agentcore.yaml` | `launch(auto_update_on_conflict=True)`, or delete the config and re-`configure` |
| Client times out on a long agent | Default read timeout too short | Raise boto3 `read_timeout` (e.g. 300s) for long-running agents |
| 4xx on `/invocations` locally | Body isn't valid JSON | Send `Content-Type: application/json` + a JSON body |

## Skeptic's corner — "why not just a Lambda?"

You can run a *simple* agent in Lambda. Runtime earns its place when you need: sessions **longer than Lambda's 15 minutes** (up to 8h), **session affinity** (same warm microVM per user), built-in **streaming**, and first-class hooks for **Identity / Memory / Gateway / Observability**. For a stateless one-shot tool call, Lambda is fine. For a stateful, multi-turn, possibly long-running agent, Runtime is the purpose-built home.


## 10. Clean up (avoid surprise charges)

```python
runtime.destroy()        # tears down the runtime, endpoint, and toolkit-created resources
```
or, on the CLI: `agentcore destroy`.

## Next

**`03_memory.ipynb`** — give TravelMind a memory that survives across sessions: short-term vs long-term, the four memory **strategies**, wiring memory into a Strands agent with a hook, and using memory **branches** to hand data between agents.
